In [ ]:
import torch
from datasets import load_dataset
from transformers import ( 
                    AutoTokenizer, 
                    AutoModelForSequenceClassification,
                    TrainingArguments, 
                    Trainer)


# 1. 데이터셋 및 토크나이저 로드
dataset = load_dataset("imdb")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# 2. 모델 설정
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

# 3. 학습 인자 설정
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    evaluation_strategy="epoch",
    num_train_epochs=3,
    fp16=True, # GPU 가속
)

# 4. Trainer 정의 및 학습
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
)

trainer.train()

In [ ]:
import os, random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

# -------------------------
# 0) 재현성/환경 고정
# -------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(42)

# -------------------------
# 1) 데이터 형태 가정
# - texts: List[str]
# - labels: List[int]
# 시험에서는 CSV/JSON 읽어서 여기 채우는 식
# -------------------------
texts_train = ["good movie", "bad film"]
labels_train = [1, 0]
texts_val = ["nice", "terrible"]
labels_val = [1, 0]

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# -------------------------
# 2) Dataset + Collate
# -------------------------
class TextClsDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        return {"text": self.texts[i], "label": int(self.labels[i])}

def collate_fn(batch, max_len=256):
    texts = [x["text"] for x in batch]
    labels = torch.tensor([x["label"] for x in batch], dtype=torch.long)

    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt",
    )
    enc["labels"] = labels
    return enc

train_ds = TextClsDataset(texts_train, labels_train)
val_ds   = TextClsDataset(texts_val, labels_val)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds, batch_size=16, shuffle=False, collate_fn=collate_fn)

# -------------------------
# 3) 모델/옵티마/스케줄러
# -------------------------
num_labels = len(set(labels_train))
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
num_epochs = 2
total_steps = num_epochs * len(train_loader)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, int(0.1 * total_steps)),
    num_training_steps=total_steps
)

# -------------------------
# 4) Metric (정답률)
# -------------------------
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0

    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch)  # out.loss / out.logits
        loss_sum += out.loss.item() * batch["labels"].size(0)

        preds = out.logits.argmax(dim=-1)
        correct += (preds == batch["labels"]).sum().item()
        total += batch["labels"].size(0)

    return {
        "val_loss": loss_sum / max(1, total),
        "val_acc": correct / max(1, total),
    }

# -------------------------
# 5) Train loop
# -------------------------
for epoch in range(1, num_epochs + 1):
    model.train()
    running = 0.0

    for step, batch in enumerate(train_loader, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch)
        loss = out.loss

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        running += loss.item()

    metrics = evaluate(model, val_loader)
    print(f"[Epoch {epoch}] train_loss={running/len(train_loader):.4f} "
          f"val_loss={metrics['val_loss']:.4f} val_acc={metrics['val_acc']:.4f}")
